# 🌟 LEDNet: Joint Low-Light Enhancement and Deblurring (ECCV 2022)
### Interactive Google Colab Demo & Benchmark

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)
![PyTorch](https://img.shields.io/badge/PyTorch-1.12%2B-ee4c2c.svg)
![License](https://img.shields.io/badge/License-MIT-blue.svg)

This notebook demonstrates **LEDNet** for simultaneously enhancing extreme low-light visibility and removing severe camera/motion blur in a single forward pass.

**Core Innovations:**
1. **Pyramid Pooling Module (PPM)**: Multi-scale receptive fields `(1, 2, 3, 6)` capture global lighting context.
2. **Curve Channel Attention (CurveCALayer)**: High-order curve estimation for adaptive dynamic range expansion without clipping highlights.
3. **Dynamic Filtering (`KernelConv2D`)**: Spatially-variant kernel prediction for fine-grained local sharp deblurring.

## ⚙️ Step 1: Hardware Check & Environment Setup

In [ ]:
# Verify GPU availability
!nvidia-smi

# Clone repository (if running directly on clean Colab)
import os
if not os.path.exists('LEDNet'):
    # Replace with your GitHub repository URL if cloned from remote
    print("Setting up local workspace...")
    !git clone https://github.com/sczhou/LEDNet.git
    %cd LEDNet
else:
    %cd LEDNet

# Install dependencies
!pip install -q basicsr torchvision opencv-python scikit-image gradio matplotlib
print("Environment ready!")

## 📥 Step 2: Download Pre-trained Model Checkpoints
Download official ECCV 2022 weights and custom fine-tuned weights.

In [ ]:
import os
os.makedirs('weights', exist_ok=True)

# Official Baseline Checkpoint (ECCV 2022)
if not os.path.exists('weights/lednet.pth'):
    print("Downloading official lednet.pth...")
    !wget -q -O weights/lednet.pth https://github.com/sczhou/LEDNet/releases/download/v0.1.0/lednet.pth

# Official Retrain Checkpoint
if not os.path.exists('weights/lednet_retrain_500000.pth'):
    print("Downloading lednet_retrain_500000.pth...")
    !wget -q -O weights/lednet_retrain_500000.pth https://github.com/sczhou/LEDNet/releases/download/v0.1.0/lednet_retrain_500000.pth

print("Checkpoints downloaded successfully:")
!ls -lh weights/

## 🚀 Step 3: Run Inference on Challenging Test Samples

In [ ]:
# Run unified inference with benchmark logging
!python inference.py --input inputs/WorstPics --output results/colab_demo --benchmark
print("Inference completed!")

## 📊 Step 4: Before vs After Side-by-Side Comparison & IQA Metrics

In [ ]:
import cv2
import glob
import numpy as np
import matplotlib.pyplot as plt

def calc_vol(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())

def calc_uer(img, th=20):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return float(np.sum(gray < th) / gray.size) * 100.0

sample_files = sorted(glob.glob('inputs/WorstPics/*.png'))[:3]

fig, axs = plt.subplots(len(sample_files), 2, figsize=(14, 5 * len(sample_files)))
if len(sample_files) == 1:
    axs = np.expand_dims(axs, axis=0)

for i, inp_path in enumerate(sample_files):
    fname = os.path.basename(inp_path)
    out_path = os.path.join('results/colab_demo', fname)
    
    img_in = cv2.imread(inp_path)
    img_out = cv2.imread(out_path)
    
    if img_in is None or img_out is None:
        continue
        
    vol_in, uer_in = calc_vol(img_in), calc_uer(img_in)
    vol_out, uer_out = calc_vol(img_out), calc_uer(img_out)
    
    rgb_in = cv2.cvtColor(img_in, cv2.COLOR_BGR2RGB)
    rgb_out = cv2.cvtColor(img_out, cv2.COLOR_BGR2RGB)
    
    axs[i, 0].imshow(rgb_in)
    axs[i, 0].set_title(f"Input: {fname}\nUER: {uer_in:.1f}% (Dark) | VoL: {vol_in:.1f} (Blurry)", fontsize=11, color='red')
    axs[i, 0].axis('off')
    
    axs[i, 1].imshow(rgb_out)
    axs[i, 1].set_title(f"Restored (LEDNet)\nUER: {uer_out:.1f}% (-{uer_in - uer_out:.1f}%) | VoL: {vol_out:.1f} (+{(vol_out/max(vol_in, 1e-4)-1)*100:.1f}%)", fontsize=11, color='green')
    axs[i, 1].axis('off')

plt.tight_layout()
plt.show()

## 🌐 Step 5: Launch Interactive Gradio Web Demo
Creates a public URL (`gradio.live`) to test with custom image uploads.

In [ ]:
# Launch Gradio with public shareable link
!python app.py --share